---
pinned_commit: fa64069d66bfc2037292e2ee2718884d40345e12
---

# Control V2 Requested Missing Diagnostics

Loads the derived diagnostics requested after the Control V2 feature audit. This notebook only reads existing JSON and Parquet artifacts; it does not rebuild feature extraction or section audit data.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "train").exists():
            return candidate
    raise FileNotFoundError("could not find repository root")


REPO_ROOT = find_repo_root()
AUDIT_DIR = REPO_ROOT / "train/artifacts/features/control_v2_audit"
MANIFEST_PATH = AUDIT_DIR / "control_v2_requested_missing_diagnostics_manifest.json"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
outputs = {name: REPO_ROOT / path for name, path in manifest["outputs"].items()}

print("repo root:", REPO_ROOT)
print("manifest:", MANIFEST_PATH)
print("source parquet:", REPO_ROOT / manifest["generated_from"])

repo root: /Users/l/projects/Mapperatorinator
manifest: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_audit/control_v2_requested_missing_diagnostics_manifest.json
source parquet: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_audit/control_v2_section_audit_8s_stride4.parquet


## Manifest Counts

In [2]:
counts = pd.Series(manifest["counts"], name="value").to_frame()
display(counts)

,value
clustered_sections,377548.000000
feature_section_hits,230081.000000
feature_sections_per_input_section,0.564628
repeat_rhythm_near_high_sections,93030.000000
review_queue_rows,273.000000
unique_affected_section_rate,0.507717
unique_affected_sections,206890.000000


## Artifact Inventory

In [3]:
inventory_rows = []
for name, path in sorted(outputs.items()):
    parquet = pq.ParquetFile(path)
    inventory_rows.append(
        {
            "artifact": name,
            "rows": parquet.metadata.num_rows,
            "columns": len(parquet.schema_arrow.names),
            "size_mb": path.stat().st_size / 1_000_000,
            "path": path.relative_to(REPO_ROOT).as_posix(),
        }
    )
inventory = pd.DataFrame(inventory_rows).sort_values("artifact").reset_index(drop=True)
display(inventory)

,artifact,rows,columns,size_mb,path
0,kmeans8_axis_dominance,17,2,0.002191,train/artifacts/features/control_v2_audit/cont...
1,kmeans8_profile,8,23,0.017182,train/artifacts/features/control_v2_audit/cont...
2,kmeans8_representatives,40,33,0.029829,train/artifacts/features/control_v2_audit/cont...
3,kmeans8_sections,377548,32,43.726579,train/artifacts/features/control_v2_audit/cont...
4,kmeans8_stability,5,4,0.003599,train/artifacts/features/control_v2_audit/cont...
5,low_confidence_high_value_feature_breakdown,12,19,0.015302,train/artifacts/features/control_v2_audit/cont...
6,low_confidence_high_value_feature_sections,230081,36,10.354762,train/artifacts/features/control_v2_audit/cont...
7,low_confidence_high_value_review_queue,273,36,0.045619,train/artifacts/features/control_v2_audit/cont...
8,low_confidence_high_value_unique_sections,206890,20,8.739357,train/artifacts/features/control_v2_audit/cont...
9,repeat_rhythm_near_high_sections,93030,27,6.252775,train/artifacts/features/control_v2_audit/cont...


## Low Confidence / High Value

In [4]:
breakdown = pd.read_parquet(outputs["low_confidence_high_value_feature_breakdown"])
display(
    breakdown[
        [
            "feature",
            "confidence_feature",
            "low_confidence_high_value_count",
            "feature_section_rate",
            "window_only_low_confidence_count",
            "peak_low_confidence_count",
        ]
    ]
)

unique_sections = pd.read_parquet(outputs["low_confidence_high_value_unique_sections"])
display(unique_sections.head(20))

review_queue = pd.read_parquet(outputs["low_confidence_high_value_review_queue"])
display(review_queue.head(30))

,feature,confidence_feature,low_confidence_high_value_count,feature_section_rate,window_only_low_confidence_count,peak_low_confidence_count
0,ln_change_rate,ln_change_confidence,198594,0.487358,188172,10422
1,jack_streak_exposure,jack_streak_confidence,19253,0.047248,19253,0
2,repeat_rhythm,repeat_confidence,4525,0.011105,4525,0
3,density_level,density_confidence,3475,0.008528,3339,136
4,density_burst,density_confidence,3309,0.008120,2782,527
5,jack_excess,jack_confidence,366,0.000898,366,0
6,chord_ratio,chord_confidence,219,0.000537,219,0
7,repeat_shift,repeat_confidence,184,0.000452,184,0
8,repeat_motion,repeat_confidence,78,0.000191,78,0
9,repeat_exact,repeat_confidence,55,0.000135,55,0


,filtered_index,beatmap_id,section_start_s,section_end_s,features,confidence_features,feature_hit_count,max_value_p95,min_confidence_mean,min_confidence_p20,min_confidence_min,min_confidence_at_feature_peak,difficulty_first,artist_first,title_first,version_first,section_center_s_first,map_duration_s_first,valid_fraction_first,control_confidence_mean_first
0,9012,1045287,76.0,84.0,"density_burst,density_level,hand_imbalance_abs...","density_confidence,hand_confidence,ln_change_c...",8,1.411606,0.000000,0.000000,0.0,0.000000,5.28,Hilight Tribe,Free Tibet (Vini Vici Remix),Enter the GOA,80.0,353.200012,1.0000,0.190380
1,2637,3572594,80.0,88.0,"hand_balance_signed,hand_imbalance_abs,jack_ex...","hand_confidence,jack_confidence,jack_streak_co...",8,0.971074,0.180218,0.000000,0.0,0.780727,4.40,Zekk,Fluctuation,ISNOT Rise & Fall.Hitsounds,84.0,136.300003,1.0000,0.431364
2,7861,560734,0.0,8.0,"density_burst,density_level,ln_change_rate,rep...","density_confidence,ln_change_confidence,repeat...",7,3.331668,0.477818,0.000000,0.0,0.780481,4.53,Various,Yolomania Vol. 3A,Ariana Grande - Baby I [Leo137] 14,4.0,192.399994,0.9375,0.458266
3,1807,2910941,140.0,148.0,"density_burst,density_level,ln_change_rate,rep...","density_confidence,ln_change_confidence,repeat...",7,2.374678,0.121011,0.000000,0.0,0.755283,4.45,"Emma Stone, Callie Hernandez, Sonoya Mizuno, J...",Someone In The Crowd,Soulmate,144.0,254.800003,1.0000,0.523183
4,7270,4741617,200.0,208.0,"density_burst,density_level,hand_imbalance_abs...","density_confidence,hand_confidence,repeat_conf...",7,2.072470,0.306611,0.000000,0.0,0.528755,4.75,blobdash & breakchild,EPiSODES,Cinematograph,204.0,304.200012,1.0000,0.335392
5,1738,3275345,44.0,52.0,"density_burst,density_level,hand_imbalance_abs...","density_confidence,hand_confidence,jack_confid...",7,2.014655,0.023837,0.000000,0.0,0.000000,3.57,nitro,kiss the sexy robot?? ultimate championship 20...,"podium """"""finish""""""""""""""",48.0,138.600006,1.0000,0.475344
6,3293,3549918,160.0,168.0,"density_burst,density_level,jack_excess,repeat...","density_confidence,jack_confidence,repeat_conf...",7,1.766482,0.244662,0.000000,0.0,0.663138,3.80,"Tyler, The Creator & Nigo","Come On, Let's Go",Punctual,164.0,195.899994,1.0000,0.395746
7,10104,1527737,156.0,164.0,"density_burst,density_level,ln_change_rate,rep...","density_confidence,ln_change_confidence,repeat...",7,1.683521,0.088350,0.000000,0.0,0.392747,3.45,X&G,Whiplash ft. josh pan (sakuraburst Remix),break out,160.0,214.600006,1.0000,0.316663
8,7270,4741617,196.0,204.0,"density_burst,density_level,hand_imbalance_abs...","density_confidence,hand_confidence,repeat_conf...",7,1.672544,0.423570,0.212407,0.0,0.574642,4.75,blobdash & breakchild,EPiSODES,Cinematograph,200.0,304.200012,1.0000,0.446591
9,126,2161435,84.0,92.0,"density_burst,density_level,ln_change_rate,rep...","density_confidence,ln_change_confidence,repeat...",7,1.628535,0.237199,0.000000,0.0,0.784298,4.50,onumi,REGRET PART TWO,ETERNAL,88.0,146.199997,1.0000,0.326271


,feature,confidence_feature,filtered_index,beatmap_id,difficulty,artist,title,version,section_start_s,section_end_s,...,peak_value,raw_at_peak,n_eff_at_peak,numerator_at_peak,denominator_at_peak,max_streak_at_peak,left_load_at_peak,right_load_at_peak,top1_freq_at_peak,pattern_variety_at_peak
0,chord_ratio,chord_confidence,5968,4297775,3.98,ShinRa-Bansho,"Mugen Shitto Gekijou ""666""",Jealousy Playhouse,64.0,72.0,...,0.806863,0.906049,11.848392,3.621250,3.99675,NaN,NaN,NaN,NaN,NaN
1,chord_ratio,chord_confidence,5968,4297775,3.98,ShinRa-Bansho,"Mugen Shitto Gekijou ""666""",Jealousy Playhouse,68.0,76.0,...,0.806863,0.906049,11.848392,3.621250,3.99675,NaN,NaN,NaN,NaN,NaN
2,chord_ratio,chord_confidence,8465,879298,5.13,yuikonnu,"Natsu no Owari, Koi no Hajimari",Mini ZenoCORE!,184.0,192.0,...,0.753223,0.803895,14.056385,3.917583,4.87325,NaN,NaN,NaN,NaN,NaN
3,chord_ratio,chord_confidence,4022,3807222,5.86,Helblinde,DEAD END,Fantasy Mythology,196.0,204.0,...,0.731914,0.768694,15.159003,4.202833,5.46750,NaN,NaN,NaN,NaN,NaN
4,chord_ratio,chord_confidence,9662,1294052,4.13,Emmanuel Macron ft. Marine Le Pen (Khaled Frea...,Poudre de Perlimpinpin,_Pillow's Keyboard powder,24.0,32.0,...,0.728239,0.778887,13.931837,3.470917,4.45625,NaN,NaN,NaN,NaN,NaN
5,chord_ratio,chord_confidence,5080,4093451,4.20,Parry Gripp,Guinea Pig Bridge,anatha's Bridge of Guinea Pigs,32.0,40.0,...,0.726199,0.741624,18.491550,4.511667,6.08350,NaN,NaN,NaN,NaN,NaN
6,chord_ratio,chord_confidence,9662,1294052,4.13,Emmanuel Macron ft. Marine Le Pen (Khaled Frea...,Poudre de Perlimpinpin,_Pillow's Keyboard powder,20.0,28.0,...,0.722013,0.768651,14.208901,3.696250,4.80875,NaN,NaN,NaN,NaN,NaN
7,chord_ratio,chord_confidence,8164,718163,5.21,xi,Blue Zenith,Zen's Black Another,168.0,176.0,...,0.723735,0.735494,19.543655,4.815834,6.54775,NaN,NaN,NaN,NaN,NaN
8,chord_ratio,chord_confidence,4619,3972157,4.19,Ardolf,Lycanthrope,Hytex's Livid,152.0,160.0,...,0.718472,0.812449,11.627983,3.091167,3.80475,NaN,NaN,NaN,NaN,NaN
9,chord_ratio,chord_confidence,5968,4297775,3.98,ShinRa-Bansho,"Mugen Shitto Gekijou ""666""",Jealousy Playhouse,60.0,68.0,...,0.726604,0.778612,13.824449,3.629500,4.66150,NaN,NaN,NaN,NaN,NaN


## Repeat Rhythm Near-High Sections

In [5]:
display(pd.Series(manifest["repeat_rhythm_summary"], name="value").to_frame())

repeat_rhythm_sections = pd.read_parquet(outputs["repeat_rhythm_near_high_sections"])
display(repeat_rhythm_sections.head(30))

,value
near_high_difficulty_mean,4.067865
near_high_rate,0.228300
near_high_sections,93030.000000
near_high_unique_maps,7699.000000
repeat_rhythm_p95_corpus_p50,0.725428
repeat_rhythm_p95_corpus_p95,0.999973
repeat_rhythm_p95_corpus_p99,1.000000
threshold,0.950000
total_sections,407491.000000


,filtered_index,beatmap_id,difficulty,artist,title,version,section_start_s,section_end_s,section_center_s,map_duration_s,...,repeat_rhythm_max,repeat_rhythm_confidence_at_peak,repeat_rhythm_peak_time_s,repeat_rhythm_peak_value,repeat_rhythm_n_eff_at_peak,repeat_rhythm_top1_freq_at_peak,repeat_rhythm_pattern_variety_at_peak,repeat_confidence_mean,repeat_confidence_p20,repeat_confidence_min
0,10629,2197979,5.39,DJ Sharpnel,Over the Fullereneshift,Second Impact 1.2x,48.0,56.0,52.0,221.500000,...,1.0,1.0,52.000000,1.0,73.397102,1.0,0.0,1.000000,1.000000,1.000000
1,205,2205172,5.78,mafumafu,I wanna be a girl,Nyaa,92.0,100.0,96.0,234.199997,...,1.0,1.0,95.099998,1.0,83.998260,1.0,0.0,1.000000,1.000000,1.000000
2,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,116.0,124.0,120.0,132.800003,...,1.0,1.0,116.000000,1.0,81.004845,1.0,0.0,0.959020,0.870889,0.789204
3,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,112.0,120.0,116.0,132.800003,...,1.0,1.0,112.000000,1.0,81.004845,1.0,0.0,0.999996,1.000000,0.999919
4,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,108.0,116.0,112.0,132.800003,...,1.0,1.0,108.300003,1.0,81.001846,1.0,0.0,1.000000,1.000000,1.000000
5,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,104.0,112.0,108.0,132.800003,...,1.0,1.0,108.300003,1.0,81.001846,1.0,0.0,1.000000,0.999999,0.999999
6,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,96.0,104.0,100.0,132.800003,...,1.0,1.0,96.000000,1.0,81.004845,1.0,0.0,1.000000,1.000000,0.999998
7,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,92.0,100.0,96.0,132.800003,...,1.0,1.0,93.599998,1.0,80.995842,1.0,0.0,1.000000,1.000000,1.000000
8,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,88.0,96.0,92.0,132.800003,...,1.0,1.0,93.599998,1.0,80.995842,1.0,0.0,0.999998,0.999999,0.999982
9,8185,713131,5.55,UNDEAD CORPORATION,The Empress scream off ver,Zenx's SHD,68.0,76.0,72.0,132.800003,...,1.0,1.0,68.000000,1.0,81.004845,1.0,0.0,1.000000,1.000000,0.999999


## Section Clustering

In [6]:
cluster_profile = pd.read_parquet(outputs["kmeans8_profile"])
cluster_axis_dominance = pd.read_parquet(outputs["kmeans8_axis_dominance"])
cluster_representatives = pd.read_parquet(outputs["kmeans8_representatives"])
cluster_stability = pd.read_parquet(outputs["kmeans8_stability"])

display(cluster_profile)
display(cluster_axis_dominance.head(17))
display(cluster_stability)
display(cluster_representatives.head(40))

,cluster,sections,difficulty_mean,map_duration_mean,density_level_mean,density_level_p90,density_burst_p90,hold_occupancy_mean,hold_occupancy_p90,ln_change_rate_mean,...,jack_excess_p90,jack_streak_exposure_p90,hand_balance_signed_mean,hand_imbalance_abs_mean,repeat_exact_mean,repeat_shift_mean,repeat_motion_mean,repeat_rhythm_mean,valid_fraction_cluster_mean,control_confidence_mean_cluster_mean
0,0,84538,3.743913,208.354109,2.588227,2.683827,0.872512,0.037178,0.074002,0.497585,...,2.467988e-19,0.094371,-0.000390,0.025861,0.061539,0.089719,0.069211,0.585207,0.999832,0.797311
1,5,63085,2.971261,198.680132,1.965064,2.134971,0.810443,0.122024,0.178450,0.984018,...,6.311194e-20,0.009112,0.000657,0.049224,0.061001,0.102707,0.069787,0.400576,0.999501,0.712565
2,2,57582,3.239479,195.503913,2.280372,2.401677,0.761466,0.065585,0.107465,0.684222,...,5.593670e-21,0.044341,-0.000132,0.032600,0.094452,0.170853,0.113660,0.635039,0.999727,0.780673
3,4,57421,4.036790,213.644820,2.547513,2.663810,0.894590,0.255212,0.331402,2.193747,...,0.000000e+00,0.115542,0.000227,0.033235,0.057725,0.084574,0.064327,0.544432,0.999933,0.891245
4,1,49950,5.119009,230.739694,3.005247,3.114576,0.922853,0.039167,0.073990,0.497172,...,0.000000e+00,0.745225,-0.000570,0.023557,0.069172,0.107033,0.079189,0.644294,0.999902,0.836050
5,7,29499,4.016239,225.832615,2.456525,2.608203,0.881898,0.093036,0.144798,0.837445,...,9.762648e-02,0.249763,0.000575,0.051476,0.077724,0.125802,0.096028,0.591452,0.999713,0.790726
6,6,23678,3.847964,218.380269,2.437253,2.589069,0.894445,0.110288,0.169916,1.005301,...,1.985256e-09,0.184417,-0.000015,0.041520,0.067323,0.103753,0.079030,0.536814,0.999824,0.798902
7,3,11795,3.517955,240.138236,2.092422,2.237897,0.673689,0.083081,0.124577,0.677564,...,8.455405e-11,0.071708,-0.003252,0.070492,0.164821,0.271420,0.197752,0.696882,0.999188,0.738587


,column,cluster_eta2
0,jack_streak_exposure_p90,0.677446
1,density_level_p90,0.525895
2,density_level_mean,0.513084
3,repeat_shift_mean,0.456892
4,repeat_motion_mean,0.440862
5,repeat_exact_mean,0.411140
6,hold_occupancy_mean,0.410518
7,ln_change_rate_mean,0.402320
8,hold_occupancy_p90,0.356896
9,jack_excess_p90,0.348857


,seed,clustered_sections,top_axes,top_axis_eta2_mean
0,1,377548,"jack_streak_exposure_p90,density_level_p90,rep...",0.522125
1,3,377548,"jack_streak_exposure_p90,density_level_p90,den...",0.510077
2,5,377548,"jack_streak_exposure_p90,density_level_p90,den...",0.510049
3,7,377548,"jack_streak_exposure_p90,density_level_p90,den...",0.522836
4,11,377548,"jack_streak_exposure_p90,density_level_p90,den...",0.510162


,filtered_index,beatmap_id,difficulty,artist,title,version,section_start_s,section_end_s,section_center_s,map_duration_s,...,hand_balance_signed_mean,hand_imbalance_abs_mean,repeat_exact_mean,repeat_shift_mean,repeat_motion_mean,repeat_rhythm_mean,cluster,pca1,pca2,representative_strategy
0,7954,590575,4.30,Yooh,Shanghai Kouchakan ~ Chinese Tea Orchid Remix,EXHAUST,68.0,76.0,72.0,111.099998,...,-0.015212,0.020812,0.066586,0.071091,0.072691,0.632602,0,1.314358,0.755520,medoid
1,8972,1030928,3.69,P*Light feat. mow*2,Hello Happiness,Hyper,20.0,28.0,24.0,103.599998,...,0.016078,0.036300,0.052283,0.065692,0.077828,0.624083,0,1.278660,0.832464,medoid
2,8294,2127786,3.69,DJ Myosuke & Noizenecio,Architecture,Envory's Hard,220.0,228.0,224.0,244.800003,...,-0.000549,0.010794,0.054372,0.074128,0.054372,0.557767,0,1.351986,1.281063,medoid
3,4873,464275,4.24,Yooh,Shanghai Kouchakan ~ Chinese Tea Orchid Remix,zukimura's INF,68.0,76.0,72.0,111.599998,...,-0.008867,0.046595,0.064953,0.072492,0.069349,0.571216,0,1.257322,0.484948,medoid
4,9888,4599505,3.10,Vremya i Steklo,Navernopotomuchto,MX,168.0,176.0,172.0,187.699997,...,-0.010960,0.017028,0.049841,0.135317,0.054282,0.588781,0,1.359408,0.613099,medoid
5,8080,711786,4.89,Yooh,LegenD.,KK's GRAVITY,84.0,92.0,88.0,120.000000,...,0.022385,0.023506,0.063408,0.072362,0.063408,0.660659,1,1.210092,2.203826,medoid
6,3830,3733702,4.65,Camellia,"senpai, notice me!",YADA!!!,220.0,228.0,224.0,276.399994,...,0.006015,0.015429,0.066750,0.135363,0.066750,0.661778,1,1.230693,1.623016,medoid
7,9858,1393843,5.31,Camellia,GHOST,x1.0,212.0,220.0,216.0,314.200012,...,0.000213,0.012066,0.054061,0.066218,0.054061,0.698920,1,1.215990,2.785530,medoid
8,1359,2877806,5.20,Kobaryo,Bookmaker,Drago's Extra,52.0,60.0,56.0,263.299988,...,0.016525,0.016525,0.061472,0.074785,0.074785,0.660326,1,1.207201,2.266437,medoid
9,10742,1932873,4.56,Kabocha,EmbryO,EXHAUST,104.0,112.0,108.0,120.000000,...,0.000712,0.010501,0.065548,0.106667,0.069484,0.681938,1,1.231328,2.010381,medoid
